# 01 — Análisis Exploratorio de Datos
Dataset: HetRec 2011 Last.fm

In [ ]:
import sys, os
from pathlib import Path

# Buscar src/ en ubicaciones conocidas
CANDIDATES = [
    '/content/music-recommender-ncf/src',
    str(Path(os.getcwd()) / '../src'),
    str(Path(os.getcwd()) / 'src'),
]
for p in CANDIDATES:
    if Path(p).exists():
        sys.path.insert(0, p)
        print('src encontrado en:', p)
        break
else:
    print('ERROR: no se encontró src/. Rutas probadas:', CANDIDATES)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

from config import DATA_DIR
print('Cargando datos desde:', DATA_DIR)

In [ ]:
artists      = pd.read_csv(DATA_DIR / 'artists.dat',            sep='\t', encoding='latin-1')
user_artists = pd.read_csv(DATA_DIR / 'user_artists.dat',       sep='\t', encoding='latin-1')
tags         = pd.read_csv(DATA_DIR / 'tags.dat',               sep='\t', encoding='latin-1')
user_tagged  = pd.read_csv(DATA_DIR / 'user_taggedartists.dat', sep='\t', encoding='latin-1')
user_friends = pd.read_csv(DATA_DIR / 'user_friends.dat',       sep='\t', encoding='latin-1')

print(f'Usuarios:      {user_artists.userID.nunique():,}')
print(f'Artistas:      {user_artists.artistID.nunique():,}')
print(f'Interacciones: {len(user_artists):,}')
print(f'Tags únicos:   {tags.tagID.nunique():,}')
print(f'Amistades:     {len(user_friends):,}')

## Sparsity de la matriz usuario-artista

In [ ]:
n_users        = user_artists["userID"].nunique()
n_artists      = user_artists["artistID"].nunique()
n_interactions = len(user_artists)
sparsity = 1 - n_interactions / (n_users * n_artists)

print(f"Usuarios:      {n_users:,}")
print(f"Artistas:      {n_artists:,}")
print(f"Interacciones: {n_interactions:,}")
print(f"Sparsity:      {sparsity:.2%}")

## Distribuciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribución reproducciones por usuario
user_plays = user_artists.groupby('userID')['weight'].sum()
axes[0,0].hist(user_plays.values, bins=50, alpha=0.7, color='steelblue')
axes[0,0].set_title('Reproducciones totales por usuario')
axes[0,0].set_xlabel('Total de plays'); axes[0,0].set_ylabel('Usuarios')

# Top 20 artistas
artists_renamed = artists.rename(columns={'id': 'artistID'}) if 'id' in artists.columns else artists
top20 = (user_artists.groupby('artistID')['weight'].sum()
                    .sort_values(ascending=False).head(20)
                    .reset_index()
                    .merge(artists_renamed[['artistID','name']], on='artistID', how='left'))
axes[0,1].barh(top20['name'], top20['weight'], color='coral')
axes[0,1].set_title('Top 20 artistas más escuchados')
axes[0,1].invert_yaxis()

# Distribución de amigos
friends_per_user = user_friends.groupby('userID').size()
axes[1,0].hist(friends_per_user.values, bins=30, alpha=0.7, color='seagreen')
axes[1,0].set_title('Amigos por usuario')
axes[1,0].set_xlabel('Número de amigos')

# Top 15 tags
merged_tags = user_tagged.merge(tags, on='tagID', how='left')
top_tags = merged_tags['tagValue'].value_counts().head(15)
axes[1,1].barh(top_tags.index[::-1], top_tags.values[::-1], color='mediumpurple')
axes[1,1].set_title('Top 15 tags más usados')

plt.tight_layout()
plt.show()

## Red social (muestra de 150 nodos)

In [ ]:
sample_edges = user_friends.sample(150, random_state=42)
G = nx.from_pandas_edgelist(sample_edges, 'userID', 'friendID')
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, k=0.5, seed=42)
nx.draw(G, pos, node_size=15, node_color='steelblue',
        edge_color='gray', alpha=0.6, width=0.5)
plt.title('Red social Last.fm — muestra de 150 conexiones')
plt.show()
print(f'Componentes conectados: {nx.number_connected_components(G)}')

## Observaciones clave
- La distribución de plays sigue una **ley de potencia** (pocos usuarios muy activos, mayoría moderados)
- Sparsity ~99% — típico de sistemas de recomendación reales
- Tags semánticos ricos disponibles (rock, pop, electronic...) → usados como features en el NCF
- Red social disponible → trabajo futuro para graph-based recommendations